# Stage 4 — Phase 5.2/5.3: Fine-Tuning Timing Probe + Training

Self-contained — all logic lives directly in these cells (no `git clone`/`src` import),
so updates just need `git pull` in your VS Code terminal + re-running the affected cell.

Starts with **Qwen3-VL** to prove the pipeline end-to-end, then repeats for InternVL3
and Molmo2-O-7B. Per the 2026-07-10 resequencing decision: trains a SINGLE combined
LoRA directly on the detection task (5.2+5.3 merged), not a separate domain-adapt-
then-adapter sequence. Task-neutral domain-adapt is deferred to end-of-Stage-4 as its
own separate checkpoint.

## 1. Mount Drive + install dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate peft pycocotools supervision \
    kagglehub kaggle qwen-vl-utils einops timm

MessageError: User cancelled dfs_ephemeral authorization

## 2. Config + shared imports

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pid_project/data"
KAGGLE_DIR = "kaggle_pid_symbols"
GUPTA_DIR = "gupta_pid"

import json, time
from pathlib import Path
import torch
from PIL import Image
from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    MaxTimeCriteria, StoppingCriteriaList,
)

ROOT = Path(DRIVE_ROOT)
kaggle_p = ROOT / KAGGLE_DIR
gupta_p = ROOT / GUPTA_DIR
print("Kaggle:", kaggle_p.exists(), "| Gupta:", gupta_p.exists())

Kaggle: True | Gupta: True


## 3. Tiling (checklist 2.3) + dataset assembly (checklist 5.1) + training-target
## formatting — all inline, no `src` import

In [ ]:
TILE_SIZE, OVERLAP = 1024, 205
KAGGLE_TILE_SIZE = 1280  # confirmed uniform (image-health check) — don't open all 30k
                          # files just to read a known-constant dimension

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            is_edge = (x1 - x0 < tile_size) or (y1 - y0 < tile_size)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1, "is_edge": is_edge})
            x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, img_w, img_h):
    parts = line.split()
    cls = parts[0]
    cx, cy, w, h = (float(v) for v in parts[1:5])
    cx, cy, w, h = cx * img_w, cy * img_h, w * img_w, h * img_h
    return cls, cx - w/2, cy - h/2, cx + w/2, cy + h/2

def boxes_intersect(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return ax0 < bx1 and ax1 > bx0 and ay0 < by1 and ay1 > by0

def tile_and_remap(img_path, label_path):
    img = Image.open(img_path)
    W, H = img.size
    tiles = compute_tile_grid(W, H)
    orig_boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            line = line.strip()
            if line:
                orig_boxes.append(yolo_line_to_xyxy(line, W, H))
    for t in tiles:
        tbox = (t["x0"], t["y0"], t["x1"], t["y1"])
        t["boxes_tile"] = []
        for cls, x0, y0, x1, y1 in orig_boxes:
            if boxes_intersect((x0, y0, x1, y1), tbox):
                t["boxes_tile"].append((cls, x0 - t["x0"], y0 - t["y0"], x1 - t["x0"], y1 - t["y0"]))
    return img, tiles, orig_boxes

def load_test_ids():
    return set(json.loads((ROOT / "test_ids.json").read_text())["test_ids"])

def collect_gupta_train_sheets():
    raw = gupta_p / "PID_Dataset" / "0__raw_data"
    items = []
    for lbl in (raw / "labels" / "train").glob("*.txt"):
        imgs = list((raw / "sheets" / "train").glob(f"{lbl.stem}.*"))
        if imgs:
            items.append((lbl.stem, imgs[0], lbl))
    return items

def collect_kaggle_pretrain_images():
    items = []
    for lbl in (kaggle_p / "labels").glob("*.txt"):
        img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
        if img_path.exists():
            items.append((lbl.stem, img_path, lbl))
    return items

def assemble_manifest(expected_gupta_train_count=72):
    test_ids = load_test_ids()
    gupta_train = collect_gupta_train_sheets()
    overlap = set(sid for sid, _, _ in gupta_train) & test_ids
    assert not overlap, f"TRAIN/TEST LEAK DETECTED: {sorted(overlap)}"
    if expected_gupta_train_count is not None:
        assert len(gupta_train) == expected_gupta_train_count, \
            f"expected {expected_gupta_train_count} Gupta train sheets, got {len(gupta_train)}"
    kaggle_items = collect_kaggle_pretrain_images()
    return {"gupta_train": gupta_train, "kaggle_pretrain": kaggle_items}

def load_kaggle_class_names():
    classes_path = ROOT / "classes.json"
    if not classes_path.exists():
        return {}
    data = json.loads(classes_path.read_text())
    return {cid: entry["kaggle_name"] for cid, entry in data["classes"].items()}

def format_boxjson_target(boxes):
    rows = [[round(b["bbox"][0]), round(b["bbox"][1]), round(b["bbox"][2]), round(b["bbox"][3]), 1.0, b["entity_type"]]
            for b in boxes]
    return json.dumps(rows)

def format_points_target(boxes, img_w, img_h):
    if not boxes:
        return ""
    parts = []
    for i, b in enumerate(boxes, start=1):
        x0, y0, x1, y1 = b["bbox"]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        parts.append(f"{i} {round(cx/img_w*1000)} {round(cy/img_h*1000)}")
    return f'<points coords="{" ".join(parts)}"></points>'

def boxes_from_label_file(label_path, img_w, img_h, source, class_names):
    boxes = []
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        cls_id, x0, y0, x1, y1 = yolo_line_to_xyxy(line, img_w, img_h)
        entity_type = "symbol" if source == "gupta" else class_names.get(cls_id, f"class_{cls_id}")
        boxes.append({"bbox": [x0, y0, x1, y1], "entity_type": entity_type})
    return boxes

def build_examples(candidate, manifest):
    class_names = load_kaggle_class_names()
    examples = []
    for sid, img_path, lbl_path in manifest["gupta_train"]:
        _img, tiles, _ = tile_and_remap(img_path, lbl_path)
        for t in tiles:
            tw, th = t["x1"] - t["x0"], t["y1"] - t["y0"]
            boxes = [{"bbox": [x0, y0, x1, y1], "entity_type": "symbol"} for _c, x0, y0, x1, y1 in t["boxes_tile"]]
            target = format_boxjson_target(boxes) if candidate in ("qwen3vl", "internvl3") else format_points_target(boxes, tw, th)
            examples.append({"image_path": img_path, "crop": [t["x0"], t["y0"], t["x1"], t["y1"]],
                              "target_text": target, "n_boxes": len(boxes), "source": "gupta"})
    for iid, img_path, lbl_path in manifest["kaggle_pretrain"]:
        w = h = KAGGLE_TILE_SIZE
        boxes = boxes_from_label_file(lbl_path, w, h, "kaggle", class_names)
        target = format_boxjson_target(boxes) if candidate in ("qwen3vl", "internvl3") else format_points_target(boxes, w, h)
        examples.append({"image_path": img_path, "crop": None, "target_text": target,
                          "n_boxes": len(boxes), "source": "kaggle"})
    return examples

def load_example_image(ex):
    img = Image.open(ex["image_path"]).convert("RGB")
    if ex["crop"] is not None:
        img = img.crop(ex["crop"])
    return img

## 4. Qwen3-VL: prompt, load, training-input builder, LoRA, timing probe

In [ ]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

SYMBOL_DETECTION_PROMPT = """You are analyzing a tile cropped from a Piping & Instrumentation \
Diagram (P&ID). Detect every symbol (valve, instrument, flange, nozzle, safety device, or \
other P&ID symbol) visible in this image.

Respond with ONLY a JSON array of arrays, no other text, no explanation. Each inner array is
exactly: [x0, y0, x1, y1, confidence, "entity_type"]

Example: [[100, 200, 150, 260, 0.95, "valve"], [400, 50, 430, 90, 0.88, "instrument"]]

Coordinates are absolute pixel coordinates in this image (top-left origin, x right, y down),
NOT normalized. confidence is a float 0.0-1.0. If you see no symbols, respond with an empty
array: []"""

def load_qwen():
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda"
    )
    print(f"Loaded {QWEN_MODEL_ID} in {time.time()-t0:.1f}s")
    print("VRAM used:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")
    return processor, model

def build_labeled_inputs(processor, model, image, prompt, target_text, image_first=True):
    image_part = {"type": "image", "image": image}
    text_part = {"type": "text", "text": prompt}
    content = [image_part, text_part] if image_first else [text_part, image_part]
    prompt_messages = [{"role": "user", "content": content}]
    prompt_inputs = processor.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    prompt_len = prompt_inputs["input_ids"].shape[1]
    full_messages = prompt_messages + [{"role": "assistant", "content": [{"type": "text", "text": target_text}]}]
    full_inputs = processor.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False,
        return_dict=True, return_tensors="pt",
    )
    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    full_inputs["labels"] = labels
    return {k: v.to(model.device) for k, v in full_inputs.items()}

def setup_lora(model, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model
    config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                         target_modules="all-linear", task_type="CAUSAL_LM")
    peft_model = get_peft_model(model, config)
    peft_model.print_trainable_parameters()
    return peft_model

def time_training_steps(processor, model, prompt, examples, n_steps=5, lr=1e-4, image_first=True):
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    step_times = []
    for i, ex in enumerate(examples[:n_steps]):
        t0 = time.time()
        img = load_example_image(ex)
        inputs = build_labeled_inputs(processor, model, img, prompt, ex["target_text"], image_first=image_first)
        out = model(**inputs)
        loss = out.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        step_time = time.time() - t0
        step_times.append(step_time)
        print(f"step {i+1}/{n_steps}: loss={loss.item():.4f} time={step_time:.2f}s")
    avg_step_time = sum(step_times) / len(step_times)
    print(f"\nMeasured avg step time: {avg_step_time:.2f}s")
    return avg_step_time

## 5. Load Qwen3-VL and build training examples

In [ ]:
processor, model = load_qwen()

manifest = assemble_manifest()
examples = build_examples("qwen3vl", manifest)
print(f"Total training examples: {len(examples)}")
n_gupta = sum(1 for e in examples if e['source'] == 'gupta')
n_kaggle = sum(1 for e in examples if e['source'] == 'kaggle')
print(f"  gupta tiles: {n_gupta} | kaggle images: {n_kaggle}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

## 6. Apply LoRA

In [ ]:
model = setup_lora(model)

## 7. Timing probe — MEASURED, not estimated

This is the number that actually matters for planning — everything in
Stage4_Checklist_Status.md before this point is a hand-wavy guess.

In [ ]:
avg_step_time = time_training_steps(processor, model, SYMBOL_DETECTION_PROMPT, examples, n_steps=5, image_first=True)

n_examples = len(examples)
for n_epochs in (1, 2, 3):
    total_s = avg_step_time * n_examples * n_epochs
    print(f"{n_epochs} epoch(s): {total_s/3600:.2f}h projected (measured {avg_step_time:.2f}s/step x {n_examples} examples)")

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False
